In [ ]:
import pandas as pd 
import plotly.express as px

In [ ]:
df = pd.read_csv('cases_deaths.csv')

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df.groupby('country')['total_cases_per_million'].sum()

In [ ]:
regions = [
    # Continents / agrégats
    'Africa', 'Asia', 'Asia excl. China', 'Europe', 'European Union (27)',
    'High-income countries', 'Low-income countries', 'Lower-middle-income countries',
    'North America', 'Oceania', 'South America', 'Upper-middle-income countries',
    'World', 'World excl. China', 'World excl. China and South Korea',
    'World excl. China, South Korea, Japan and Singapore',
    # Territoires US
    'American Samoa', 'Guam', 'Northern Mariana Islands', 'Puerto Rico',
    'United States Virgin Islands',
    # Territoires UK
    'Anguilla', 'Bermuda', 'British Virgin Islands', 'Cayman Islands',
    'Falkland Islands', 'Gibraltar', 'Montserrat', 'Pitcairn',
    'Saint Helena', 'Turks and Caicos Islands',
    # Dépendances de la Couronne
    'Guernsey', 'Isle of Man', 'Jersey',
    # Territoires FR
    'French Guiana', 'French Polynesia', 'Guadeloupe', 'Martinique',
    'Mayotte', 'New Caledonia', 'Reunion', 'Saint Barthelemy',
    'Saint Martin (French part)', 'Saint Pierre and Miquelon', 'Wallis and Futuna',
    # Territoires NL
    'Aruba', 'Bonaire Sint Eustatius and Saba', 'Curacao', 'Sint Maarten (Dutch part)',
    # Autres territoires
    'Cook Islands', 'Faroe Islands', 'Greenland', 'Kosovo', 'Niue',
    'Palestine', 'Tokelau',
]

mask = df['country'].isin(regions)
df_region = df[mask].copy()
df_country = df[~mask].copy()

print(f"df_country: {df_country.shape} ({df_country['country'].nunique()} pays)")
print(f"df_region: {df_region.shape} ({df_region['country'].nunique()} régions/territoires)")

In [ ]:
df_country.country.unique().size

In [ ]:
df_country.groupby('country')['new_cases_per_million'].sum()

In [ ]:
# Total des cas par million à la dernière date pour chaque pays
last_date = df_country.groupby('country')['date'].max()
df_last = df_country.merge(last_date.rename('last_date'), on='country')
df_last = df_last[df_last['date'] == df_last['last_date']]

fig = px.histogram(df_last, x='total_cases_per_million', nbins=30,
                   labels={'total_cases_per_million': 'Total cases per million'},
                   title='Distribution du total de cas par million (dernière date)')
fig.update_layout(yaxis_title='Nombre de pays')
fig.show()

In [ ]:
COUNTRY = 'Benin'

In [ ]:
df_country_selected = df_country[df_country['country'] == COUNTRY].copy()
df_country_selected['date'] = pd.to_datetime(df_country_selected['date'])
df_country_selected = df_country_selected.set_index('date').sort_index()
df_country_week = df_country_selected[['new_cases_per_million']].resample('W').sum()

fig = px.line(df_country_week, y='new_cases_per_million',
              labels={'date': 'Date', 'new_cases_per_million': 'Nouveaux cas par million (par semaine)'},
              title=f'Évolution du COVID-19 en {COUNTRY} (nouveaux cas par million, pas hebdomadaire)')
fig.show()